In [0]:
%python
from pyspark.sql import Row
import random

random.seed(42)

regions = ["North", "South", "East", "West"]

categories = ["Electronics", "Grocery", "Clothing", "Furniture"]

rows = []
for i in range(1, 201):
    rows.append(Row(
        order_id=1,
        region = random.choice(regions),
        product_category = random.choice(categories),
        amount = round(random.uniform(10, 500), 2),
        order_date = f"2026-0{random.randint(1,9)}-{random.randint(10, 28)}"
    ))


sales_df = spark.createDataFrame(rows)
sales_df.show(5)
sales_df.printSchema()

In [0]:
%sql
CREATE DATABASE IF NOT EXISTS retail;

In [0]:
%python
spark.sql("USE retail")

In [0]:
%python
sales_df.write.format("delta").mode("overwrite").saveAsTable("retail.bronze_sales")

In [0]:
%python
from pyspark.sql import functions as F

silver_df = (spark.table("retail.bronze_sales")
             .dropDuplicates(["order_id"])
             .filter("amount > 0")
             .withColumn("order_date", F.to_date(F.col("order_date"))))
silver_df.write.format("delta").mode("overwrite").saveAsTable("retail.silver_sales")

In [0]:
%python
gold_df = (spark.table("retail.silver_sales")
           .groupBy("region", "product_category")
           .agg(F.sum("amount").alias("total_revenue"),
                F.count("order_id").alias("num_orders")))

gold_df.write.format("delta").mode("overwrite").saveAsTable("retail.gold_sales_summary")


In [0]:
%sql
SELECT * FROM retail.gold_sales_summary ORDER BY total_revenue DESC;
 

In [0]:
%sql
SELECT region, SUM(total_revenue) AS region_total
FROM retail.gold_sales_summary
GROUP BY region
ORDER BY region_total DESC;

In [0]:
from delta.tables import DeltaTable

dt = DeltaTable.forName(spark, "retail.bronze_sales")
dt.history().select("version", "operation", "timestamp").show()

In [0]:
updates = spark.createDataFrame([(1, "North", "Electronics", 999.99, "2026-09-01")],
    ["order_id", "region", "product_category", "amount", "order_date"])

target = DeltaTable.forName(spark, "retail.bronze_sales")
target.alias("t").merge(
    updates.alias("s"), "t.order_id = s.order_id"
).whenMatchedUpdateAll() \
 .whenNotMatchedInsertAll() \
 .execute()

spark.sql("SELECT * FROM retail.bronze_sales WHERE order_id = 1").show()

In [0]:
spark.sql("SELECT * FROM retail.bronze_sales VERSION AS OF 0 WHERE order_id = 1").show()

In [0]:
%sql
OPTIMIZE retail.bronze_sales ZORDER BY (region);
VACUUM retail.bronze_sales RETAIN 168 HOURS;